# Fine-tuned YOLO11 evaluation and comparison
Evaluates `best.pt` once on the same frozen custom test split as the baseline, then compares recorded metrics. This notebook makes no quality claim until it has actually run.

## Config

In [ ]:
RUN_NAME = 'yolo11n_mio_v1'
BEST_CHECKPOINT = None  # Set explicit path to override inferred outputs/mio_tcd/train/<RUN_NAME>/weights/best.pt
IMG_SIZE = 640
DEVICE = None
BATCH_SIZE = 16
WORKERS = 4
PROJECT_DIR = 'outputs/mio_tcd/finetuned_eval'
EVAL_NAME = 'best_on_frozen_test'
FORCE_OVERWRITE = False

## Imports and validation

In [ ]:
from pathlib import Path
import sys, pandas as pd
from ultralytics import YOLO
sys.path.insert(0, str(Path.cwd()))
from mio_tcd_utils import PROJECT_ROOT, metric_tables
DATA_YAML = PROJECT_ROOT / 'data/mio_tcd/yolo/mio_tcd.yaml'; TEST_LIST = PROJECT_ROOT / 'data/mio_tcd/splits/test.txt'
checkpoint = Path(BEST_CHECKPOINT) if BEST_CHECKPOINT else PROJECT_ROOT / 'outputs/mio_tcd/train' / RUN_NAME / 'weights/best.pt'
if not DATA_YAML.is_file() or not TEST_LIST.is_file() or not checkpoint.is_file(): raise FileNotFoundError('Need prepared data, frozen test and fine-tuned best.pt.')
print('Evaluating frozen test:', TEST_LIST, '\nCheckpoint:', checkpoint)

## Evaluate and save metrics

In [ ]:
run_dir = PROJECT_ROOT / PROJECT_DIR
if (run_dir / EVAL_NAME).exists() and not FORCE_OVERWRITE: raise FileExistsError('Evaluation run exists; choose a new EVAL_NAME or set FORCE_OVERWRITE=True.')
results = YOLO(str(checkpoint)).val(data=str(DATA_YAML), split='test', imgsz=IMG_SIZE, batch=BATCH_SIZE, workers=WORKERS, device=DEVICE, project=str(run_dir), name=EVAL_NAME, exist_ok=FORCE_OVERWRITE, plots=True)
overall, per_class = metric_tables(results, checkpoint.name); display(overall); display(per_class)
metrics_out = PROJECT_ROOT / 'outputs/mio_tcd/finetuned_metrics.csv'; metrics_out.parent.mkdir(parents=True, exist_ok=True)
if metrics_out.exists() and not FORCE_OVERWRITE: raise FileExistsError(f'{metrics_out} exists; set FORCE_OVERWRITE=True to replace it.')
overall.to_csv(metrics_out, index=False); per_class.to_csv(metrics_out.with_name('finetuned_per_class_metrics.csv'), index=False)

## Compare with baseline, if available

In [ ]:
baseline_path = PROJECT_ROOT / 'outputs/mio_tcd/baseline_metrics.csv'
if baseline_path.is_file():
    baseline = pd.read_csv(baseline_path).iloc[0]; tuned = overall.iloc[0]
    metrics = ['precision', 'recall', 'mAP50', 'mAP50-95', 'latency_ms']
    comparison = pd.DataFrame({'metric': metrics, 'baseline': [baseline[m] for m in metrics], 'finetuned': [tuned[m] for m in metrics]})
    comparison['delta'] = comparison.finetuned - comparison.baseline
    display(comparison); comparison.to_csv(PROJECT_ROOT / 'outputs/mio_tcd/model_comparison.csv', index=False)
else: print('Baseline metrics not found; fine-tuned metrics were saved without a comparison.')
display(per_class[per_class['class'].isin(['bicycle', 'motorcycle'])])